# Exploitation de données électorales avec Python

Antoine Rustenholz, Aziz Seghaier, Jasmin Neveu
31.04.2026

In [ ]:
# On commence par importer les données depuis l'URL demandée.
import pandas as pd
df_main = pd.read_csv(
 'https://www.data.gouv.fr/fr/datasets/r/182268fc-2103-4bcb-a850-6cf90b02a9eb'
)

In [ ]:
# On en crée une copie afin d'éviter de multiplier les imports.
df = df_main.copy()

# Commençons par étudier nos données.
print(df.describe)
print(df.dtypes)
print(df.isna().sum())

In [ ]:
# On remarque que seule la variable prenom contient des valeurs manquantes.
# Ces valeurs correspondent aux votes non exprimés (abstentions, blancs ou nuls).
# C'est pourquoi on remplace ces valeurs par la nature du vote non exprimé, à savoir la valeur de la variable nom.
df['prenom'] = df['prenom'].fillna(df['nom'])
print(df.isna().sum())

# On remarque que toutes les variables sont de type object sauf code_commune et voix qui sont de type int64.
# On convertit la variable code_commune en type object.
df['code_commune'] = df['code_commune'].astype(object)
print(df.dtypes)


## 1. Explorations générales

<div class="callout callout-style-default callout-tip callout-titled">
<div class="callout-header d-flex align-content-center">
<div class="callout-icon-container">
<i class="callout-icon"></i>
</div>
<div class="callout-title-container flex-fill">
Question 1
</div>
</div>
<div class="callout-body-container callout-body">

Créer ou mettre à jour les variables suivantes:

-   `code_commune`: En utilisant la variable déjà existante et le département, remplacer la valeur `code_commune` pour constituer un vrai code commune. Par exemple, pour Montrouge, vous devriez obtenir 92049.

-   `candidat`: créer une colonne avec le prenom et le nom mis ensemble, en n’oubliant pas de mettre un espace. Ne pas éliminer les bulletins abstentions, blancs ou nuls, nous allons les exploiter ultérieurement.

</div>
</div>

In [ ]:
# Pour créer un vrai code commune, il suffit de concaténer les codes du départements et de la commune
# en ajoutant des zéros afin d'avoir toujours un code à 5 chiffres.
df['code_commune'] = (
    df['code_departement'].astype(str).str.zfill(2) +
    df['code_commune'].astype(str).str.zfill(3)
)

import numpy as np
# On fait juste attention aux votes non exprimés.
df['candidat'] = np.where(
    df['prenom'] == df['nom'],
    df['prenom'],
    df['prenom'] + ' ' + df['nom']
)

df.sample(10)

<div class="callout callout-style-default callout-tip callout-titled">
<div class="callout-header d-flex align-content-center">
<div class="callout-icon-container">
<i class="callout-icon"></i>
</div>
<div class="callout-title-container flex-fill">
Question 2
</div>
</div>
<div class="callout-body-container callout-body">

Compléter la phrase suivante grâce à Python:
En 2022, il y avait XXXXX candidats à l’élection présidentielle.
*Note: Attention aux votes non exprimés et aux abstentions*

</div>
</div>

In [ ]:
# Il suffit de compter le nombre de valeurs différentes pour candidats en supprimant les abstentions, les votes blancs et nuls.
f"En 2022, il y avait {df['candidat'].nunique()-3} candidats à l'élection présidentielle."

<div class="callout callout-style-default callout-tip callout-titled">
<div class="callout-header d-flex align-content-center">
<div class="callout-icon-container">
<i class="callout-icon"></i>
</div>
<div class="callout-title-container flex-fill">
Question 3
</div>
</div>
<div class="callout-body-container callout-body">

Calculer les scores nationaux de chaque candidat. Représenter dans ce tableau, pour chaque candidat, le nombre de voix et le pourcatage des votes exprimés (c’est-à-dire en retirant abstentions et votes non exprimés).
Représenter cela dans un *dataframe* ou, pour avoir tous les points, dans un tableau mis en forme via `great_tables` (il n’est pas obligatoire d’aller aussi loin dans la mise en forme mais essayez d’obtenir un beau tableau tout de même).
*Note: vous pouvez contrôler vos résultats obtenus avec cette page.*

</div>
</div>

In [ ]:
# Création d'une table auxiliaire
resultats = (
    df[['candidat', 'voix']]
    .groupby('candidat', as_index = False).sum()
)

# Suppression des non-candidats
resultats = resultats[~resultats['candidat'].isin(['abstentions', 'nuls', 'blancs'])]

# Calcul du nombre total de voix exprimées
nombre_total = resultats['voix'].sum()

# Création de la variable score
resultats['score'] = resultats['voix'] / nombre_total

# Tri
resultats = resultats.sort_values('voix', ascending=False)


# Génération du tableau
from great_tables import GT

(
    GT(resultats)
    .tab_header(
        title = "Elections présidentielles 2022",
        subtitle = "Résultats du premier tour (10 avril 2022)"
    )
    .cols_label( # Renommage des colonnes
        candidat = "Candidat",
        voix = "Nombre votes (total)",
        score = "Score (% votes exprimés)"
    )
    .fmt_number( # Formatage des nombres de voix
        columns = "voix",
        decimals = 0,
        sep_mark = ' '
    )
    .fmt_percent( # Formatage du score en pourcentage
        columns = "score",
        decimals = 2
    )
)

## 2. Comparaison des scores départements aux moyennes nationales.

<div class="callout callout-style-default callout-tip callout-titled">
<div class="callout-header d-flex align-content-center">
<div class="callout-icon-container">
<i class="callout-icon"></i>
</div>
<div class="callout-title-container flex-fill">
Question 4
</div>
</div>
<div class="callout-body-container callout-body">

Créer un *dataframe* nommé `score_departements` stockant, pour chaque département, le nombre de vote obtenu pour chaque candidat et le score (en %).

</div>
</div>

In [ ]:

df_exprimes = df[~df['candidat'].isin([' abstentions', ' nuls', ' blancs'])].copy()
votes_par_dept = (
    df_exprimes
    .groupby(['code_departement', 'candidat'], as_index=False)['voix']
    .sum()
)
score_departements = votes_par_dept.copy()
score_departements = score_departements.rename(columns={'voix': 'votes_departements'})
total_par_dept = (
    score_departements
    .groupby('code_departement')['votes_departements']
    .transform('sum')
)
score_departements['score_departement'] = (
    score_departements['votes_departements'] / total_par_dept
)

score_departements_display = score_departements.copy()
score_departements_display = score_departements_display.rename(columns={
    'code_departement': 'Département',
    'candidat': 'Candidat',
    'votes_departements': 'Voix département',
    'score_departement': 'Score (%)'
})
(
    GT(score_departements_display.head(20))
    .tab_header(
        title="Scores par département",
        subtitle="Premier tour 2022"
    )
    .fmt_number(
        columns="Voix département",
        decimals=0,
        sep_mark=' '
    )
    .fmt_percent(
        columns="Score (%)",
        decimals=2
    )
)



<div class="callout callout-style-default callout-tip callout-titled">
<div class="callout-header d-flex align-content-center">
<div class="callout-icon-container">
<i class="callout-icon"></i>
</div>
<div class="callout-title-container flex-fill">
Question 5
</div>
</div>
<div class="callout-body-container callout-body">

Refaire le lien avec le niveau national pour comparer le score départemental avec le score national. Nommer ce *dataframe* `score_departements`, nous allons le réutiliser par la suite.

</div>
</div>

In [ ]:
candidats_liste = df_exprimes['candidat'].unique().tolist()
votes_nationaux = (
    df_exprimes
    .groupby('candidat', as_index=False)['voix']
    .sum()
    .rename(columns={'voix': 'votes_nationales'})
)
total_national = votes_nationaux['votes_nationales'].sum()
votes_nationaux['score_national'] = votes_nationaux['votes_nationales'] / total_national
score_departements = score_departements.merge(
    votes_nationaux[['candidat', 'votes_nationales', 'score_national']],
    on='candidat',
    how='left'
)
score_departements_display = score_departements.rename(columns={
    'code_departement': 'Département',
    'candidat': 'Candidat',
    'votes_departements': 'Voix département',
    'score_departement': 'Score (%)',
    'votes_nationales': 'Voix nationales',
    'score_national': 'Score national (%)'
})
(
    GT(score_departements_display.head(20))
    .tab_header(
        title="Scores départementaux vs nationaux",
        subtitle="Premier tour 2022"
    )
    .fmt_number(
        columns=["Voix département", "Voix nationales"],
        decimals=0,
        sep_mark=' '
    )
    .fmt_percent(
        columns=["Score (%)", "Score national (%)"],
        decimals=2
    )
)


<div class="callout callout-style-default callout-tip callout-titled">
<div class="callout-header d-flex align-content-center">
<div class="callout-icon-container">
<i class="callout-icon"></i>
</div>
<div class="callout-title-container flex-fill">
Question 6
</div>
</div>
<div class="callout-body-container callout-body">

Créer une variable `surrepresentation` qui compare, en relatif, les scores nationaux et départementaux.
Par exemple, si un candidat a un score de 30% dans un département mais de 15% ailleurs, la valeur de `surrepresentation` sera égale à 100 (%).


</div>
</div>

In [ ]:
score_departements['surrepresentation'] = (
    (score_departements['score_departement'] - score_departements['score_national'])
    / score_departements['score_national'] * 100
)
score_departements_display = score_departements.rename(columns={
    'code_departement': 'Département',
    'candidat': 'Candidat',
    'votes_departements': 'Voix département',
    'score_departement': 'Score (%)',
    'votes_nationales': 'Voix nationales',
    'score_national': 'Score national (%)',
    'surrepresentation': 'Surrépresentation (%)'
})
(
    GT(score_departements_display.head(20))
    .tab_header(
        title="Surrépresentation par département",
        subtitle="Écart relatif au score national (%) - Premier tour 2022"
    )
    .fmt_number(
        columns=["Voix département", "Voix nationales"],
        decimals=0,
        sep_mark=' '
    )
    .fmt_percent(
        columns=["Score (%)", "Score national (%)"],
        decimals=2
    )
    .fmt_number(
        columns="Surrépresentation (%)",
        decimals=1,
        sep_mark=' '
    )
)

<div class="callout callout-style-default callout-tip callout-titled">
<div class="callout-header d-flex align-content-center">
<div class="callout-icon-container">
<i class="callout-icon"></i>
</div>
<div class="callout-title-container flex-fill">
Question 7
</div>
</div>
<div class="callout-body-container callout-body">

Créer une fonction pour représenter une figure similaire à Figure 1 pour un candidat donné des
principales surreprésentations (en valeur absolue) par département.

</div>
</div>

In [ ]:
import matplotlib.pyplot as plt
def plot_surrepresentation(candidat, n=5):
    df_candidat = score_departements[score_departements['candidat'] == candidat].copy()
    df_candidat = df_candidat.sort_values('surrepresentation', key=abs, ascending=False).head(n)
    
    plt.figure(figsize=(10, 6))
    colors = ['#2ecc71' if x >= 0 else '#e74c3c' for x in df_candidat['surrepresentation']]
    
    plt.barh(df_candidat['code_departement'], df_candidat['surrepresentation'], color=colors)
    plt.xlabel('Surrépresentation (%)')
    plt.ylabel('Département')
    plt.title(f'Top {n} des surreprésentations de {candidat}')
    plt.axvline(x=0, color='black', linewidth=0.5)
    plt.tight_layout()
    plt.show()
# Exemple avec Éric ZEMMOUR
plot_surrepresentation('Éric ZEMMOUR')

## 3. Un peu de cartographie

<div class="callout callout-style-default callout-tip callout-titled">
<div class="callout-header d-flex align-content-center">
<div class="callout-icon-container">
<i class="callout-icon"></i>
</div>
<div class="callout-title-container flex-fill">
Question 8
</div>
</div>
<div class="callout-body-container callout-body">

Faire une fonction permettant de restreindre `score_departements` en fonction d’un candidat. Commencer par tester sur Marine Le Pen (créer un nouvel objet, ne pas écraser `score_departements` nous allons l’utiliser à nouveau !).
Faire une jointure au fond de carte des départements et effectuer une carte de la représentation.

</div>
</div>